## Parameterized pgbench Test for Databricks Jobs
This notebook runs pgbench tests with configurable parameters for use in Databricks Jobs.


In [ ]:
## Setup Requirements
%pip install --upgrade databricks-sdk


In [ ]:
dbutils.library.restartPython()


## Install pgbench


In [ ]:
# %sh
# apt-get update && apt-get install -y wget gnupg lsb-release
# sh -c 'echo "deb http://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" \
#     > /etc/apt/sources.list.d/pgdg.list'
# wget --quiet -O - https://www.postgresql.org/media/keys/ACCC4CF8.asc | apt-key add -
# apt-get update
# apt-get install -y postgresql-client-15


In [ ]:
# %sh
# apt-get install -y postgresql-contrib-15


In [ ]:
# %sh
# pgbench --version


## Parse Job Parameters


In [ ]:
import os, subprocess, re, glob, numpy as np, json
from databricks.sdk import WorkspaceClient
import uuid, shutil as _shutil

# Parse job parameters with defaults
def get_param(param_name, default_value, param_type=str):
    """Get parameter from dbutils.widgets or use default"""
    try:
        value = dbutils.widgets.get(param_name)
        if value:
            if param_type == int:
                return int(value)
            elif param_type == float:
                return float(value)
            elif param_type == bool:
                return value.lower() in ['true', '1', 'yes']
            else:
                return value
    except:
        pass
    return default_value

# Job parameters
LAKEBASE_INSTANCE_NAME = get_param("lakebase_instance_name", "ak-lakebase-accelerator-instance")
DATABASE_NAME = get_param("database_name", "databricks_postgres")
PGBENCH_CLIENTS = get_param("pgbench_clients", 8, int)
PGBENCH_JOBS = get_param("pgbench_jobs", 8, int)
PGBENCH_DURATION = get_param("pgbench_duration", 30, int)
PGBENCH_PROGRESS_INTERVAL = get_param("pgbench_progress_interval", 5, int)
PGBENCH_PROTOCOL = get_param("pgbench_protocol", "prepared")
PGBENCH_PER_STATEMENT_LATENCY = get_param("pgbench_per_statement_latency", True, bool)
PGBENCH_DETAILED_LOGGING = get_param("pgbench_detailed_logging", True, bool)
PGBENCH_CONNECT_PER_TRANSACTION = get_param("pgbench_connect_per_transaction", False, bool)

# Query configuration (JSON string)
QUERY_CONFIG_JSON = get_param("query_config", '[]')

print(f"Parameters:")
print(f"  Lakebase Instance: {LAKEBASE_INSTANCE_NAME}")
print(f"  Database: {DATABASE_NAME}")
print(f"  Clients: {PGBENCH_CLIENTS}")
print(f"  Jobs: {PGBENCH_JOBS}")
print(f"  Duration: {PGBENCH_DURATION}s")
print(f"  Protocol: {PGBENCH_PROTOCOL}")
print(f"  Query Config: {QUERY_CONFIG_JSON}")


## Setup Connection and Queries


In [ ]:
# -------------------------
# 1) Connection env
# -------------------------

# Unified connection setup using PostgreSQL credentials
print("🔧 Setting up database connection")

env = os.environ.copy()

# Get PostgreSQL connection parameters
PGHOST = get_param("pghost", "")
PGPORT = get_param("pgport", "5432")
PGUSER = get_param("pguser", "")
PGPASSWORD = get_param("pgpassword", "")
PGSSLMODE = get_param("pgsslmode", "require")
# Optional libpq options (e.g. "-c search_path=<schema>,public") so unqualified
# table names in the scripts resolve to a chosen (e.g. synced) schema.
PGOPTIONS = get_param("pgoptions", "")

if not PGHOST or not PGUSER or not PGPASSWORD:
    raise ValueError("PostgreSQL credentials required: pghost, pguser, and pgpassword")

env.update({
    "PGHOST": PGHOST,
    "PGPORT": PGPORT,
    "PGDATABASE": DATABASE_NAME,
    "PGUSER": PGUSER,
    "PGPASSWORD": PGPASSWORD,
    "PGSSLMODE": PGSSLMODE,
})
if PGOPTIONS:
    env["PGOPTIONS"] = PGOPTIONS

print(f"  PGHOST: {PGHOST}")
print(f"  PGUSER: {PGUSER}")
print(f"  PGDATABASE: {DATABASE_NAME}")
if PGOPTIONS:
    print(f"  PGOPTIONS: {PGOPTIONS}")

print("pgbench at:", _shutil.which("pgbench"))

# -------------------------
# 2) Parse query configuration and write scripts locally
# -------------------------
workdir = "/databricks/driver/pgbench_mix"

# Clean workdir to remove old log files from previous runs
if os.path.exists(workdir):
    import shutil
    shutil.rmtree(workdir)
    print(f"Cleaned old workdir: {workdir}")

os.makedirs(workdir, exist_ok=True)
print(f"Created fresh workdir: {workdir}")

try:
    query_configs = json.loads(QUERY_CONFIG_JSON)
except json.JSONDecodeError:
    print("Invalid query config JSON, using default queries")
    query_configs = []

# Default queries if none provided. These mirror the app's 5 sample OLTP queries
# against a synced samples.tpcds_sf1.store_sales table (target table: store_sales).
if not query_configs:
    query_configs = [
        {
            "name": "order_lookup",
            "content": "\\set ticket random(1, 240000)\nSELECT ss_item_sk, ss_net_paid, ss_quantity\nFROM store_sales\nWHERE ss_ticket_number = :ticket;",
            "weight": 40
        },
        {
            "name": "customer_history",
            "content": "\\set cust random(1, 100000)\nSELECT ss_ticket_number, ss_sold_date_sk, ss_net_paid\nFROM store_sales\nWHERE ss_customer_sk = :cust\nORDER BY ss_sold_date_sk DESC\nLIMIT 50;",
            "weight": 25
        },
        {
            "name": "item_sales_agg",
            "content": "\\set item random(1, 18000)\nSELECT count(*) AS lines, sum(ss_quantity) AS units, avg(ss_sales_price) AS avg_price\nFROM store_sales\nWHERE ss_item_sk = :item;",
            "weight": 15
        },
        {
            "name": "store_daily_revenue",
            "content": "\\set store random(1, 10)\n\\set start random(2450816, 2452442)\nSELECT ss_sold_date_sk, count(*) AS tickets, sum(ss_net_paid) AS revenue\nFROM store_sales\nWHERE ss_store_sk = :store AND ss_sold_date_sk BETWEEN :start AND :start + 200\nGROUP BY ss_sold_date_sk\nORDER BY ss_sold_date_sk;",
            "weight": 12
        },
        {
            "name": "top_items_by_store",
            "content": "\\set store random(1, 10)\nSELECT ss_item_sk, sum(ss_net_paid) AS revenue, sum(ss_quantity) AS units\nFROM store_sales\nWHERE ss_store_sk = :store\nGROUP BY ss_item_sk\nORDER BY revenue DESC\nLIMIT 20;",
            "weight": 8
        }
    ]

# Write query files
query_files = []
for query_config in query_configs:
    query_name = query_config.get("name", "query")
    query_content = query_config.get("content", "")
    
    query_path = os.path.join(workdir, f"{query_name}.sql")
    with open(query_path, "w") as f:
        f.write(query_content.strip() + "\n")
    
    query_files.append((query_path, query_config.get("weight", 1)))
    print(f"Created query file: {query_path}")

# Verify all files exist
for query_path, _ in query_files:
    assert os.path.exists(query_path), f"Missing script: {query_path}"

## Execute pgbench Test


In [ ]:
# -------------------------
# 3) Build pgbench command
# -------------------------
cmd = [
    "pgbench",
    "-n",  # no vacuuming
    "-c", str(PGBENCH_CLIENTS),
    "-j", str(PGBENCH_JOBS),
    "-T", str(PGBENCH_DURATION),
    "-P", str(PGBENCH_PROGRESS_INTERVAL),
    "-M", PGBENCH_PROTOCOL,
]

# Add optional flags
if PGBENCH_PER_STATEMENT_LATENCY:
    cmd.append("-r")

if PGBENCH_DETAILED_LOGGING:
    cmd.append("-l")

if PGBENCH_CONNECT_PER_TRANSACTION:
    cmd.append("-C")

# Add query files with weights using pgbench @weight syntax
for query_path, weight in query_files:
    cmd.extend(["-f", f"{query_path}@{int(weight)}"])

# print(f"pgbench command: {' '.join(cmd)}")

# -------------------------
# 4) Run & parse output
# -------------------------
def _blks():
    """(blks_hit, blks_read) for the current DB via psql; None on any failure."""
    try:
        out = subprocess.run(
            ["psql", "-tA", "-F", "|", "-c",
             "SELECT blks_hit, blks_read FROM pg_stat_database WHERE datname = current_database()"],
            capture_output=True, text=True, env=env, timeout=30,
        ).stdout.strip()
        hit, read = out.split("|")
        return int(hit), int(read)
    except Exception:
        return None

cache_before = _blks()
print("\n=== Starting pgbench test ===")
res = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=workdir)
cache_after = _blks()

def _cache_hit_pct(before, after):
    if not before or not after:
        return None
    d_hit = max(0, after[0] - before[0])
    d_read = max(0, after[1] - before[1])
    total = d_hit + d_read
    return round(d_hit * 100.0 / total, 2) if total > 0 else None

cache_hit_pct = _cache_hit_pct(cache_before, cache_after)

print("=== STDOUT ===\n", res.stdout)
if res.stderr:
    print("=== STDERR ===\n", res.stderr)

if res.returncode != 0:
    raise SystemExit(f"pgbench failed (exit {res.returncode}). See STDERR above. Workdir: {workdir}")

# Parse TPS
m = re.search(r"tps\s*=\s*([\d\.]+)", res.stdout)
tps = float(m.group(1)) if m else None
print(f"\n=== RESULTS ===")
print(f"TPS: {tps}")

# Parse latencies from log files
# pgbench log format (with -l): client_id transaction_no time script_no time_epoch time_us [schedule_lag]
# Column 2 (time) is the transaction latency in MICROSECONDS - need to convert to milliseconds!
latencies = []
per_script = {}
log_files = glob.glob(os.path.join(workdir, "pgbench_log.*"))
print(f"\nDEBUG: Found {len(log_files)} log files: {log_files}")

for path in log_files:
    line_count = 0
    with open(path) as f:
        for line_num, line in enumerate(f, 1):
            parts = line.split()
            if len(parts) >= 3:  # Need at least client_id, transaction_no, time
                try:
                    # Column index 2 is 'time' - the transaction latency in microseconds
                    latency_us = float(parts[2])
                    latency_ms = latency_us / 1000.0  # Convert µs to ms
                    latencies.append(latency_ms)
                    if len(parts) >= 4:
                        try:
                            per_script.setdefault(int(parts[3]), []).append(latency_ms)
                        except ValueError:
                            pass
                    line_count += 1
                    
                    # Print first line of first file for debugging
                    if len(latencies) == 1:
                        print(f"DEBUG: First log line: {line.strip()}")
                        print(f"DEBUG: Parsed parts: {parts}")
                        print(f"DEBUG: parts[2] (latency in µs): {parts[2]} µs = {latency_ms} ms")
                except (ValueError, IndexError) as e:
                    if line_num <= 3:  # Only print errors from first few lines
                        print(f"DEBUG: Error parsing line {line_num}: {e} | Line: {line.strip()}")
    print(f"DEBUG: Parsed {line_count} latencies from {os.path.basename(path)}")

print(f"DEBUG: Total latencies collected: {len(latencies)}")

if latencies:
    p50, p95, p99 = (float(v) for v in np.percentile(latencies, [50, 95, 99]))
    print(f"Latency p50/p95/p99 (ms): {p50:.3f} / {p95:.3f} / {p99:.3f}")
    print(f"Total transactions: {len(latencies)}")
else:
    print("No pgbench_log.* found or no latencies parsed.")

print(f"\nLogs & scripts available at: {workdir}")

# -------------------------
# 5) Store results for job output
# -------------------------
# Per-query (per-script) breakdown, mapped to query names by -f order.
per_query = []
for _script_no, _lat in per_script.items():
    _name = query_configs[_script_no]["name"] if _script_no < len(query_configs) else f"script {_script_no}"
    _p95, _p99 = (float(v) for v in np.percentile(_lat, [95, 99]))
    per_query.append({
        "query_identifier": _name,
        "calls": len(_lat),
        "avg_time_ms": round(sum(_lat) / len(_lat), 3),
        "total_time_ms": round(sum(_lat), 3),
        "p95_time_ms": round(_p95, 3),
        "p99_time_ms": round(_p99, 3),
    })
per_query.sort(key=lambda d: d["total_time_ms"], reverse=True)

results = {
    "test_parameters": {
        "lakebase_instance": LAKEBASE_INSTANCE_NAME,
        "database_name": DATABASE_NAME,
        "clients": PGBENCH_CLIENTS,
        "jobs": PGBENCH_JOBS,
        "duration_seconds": PGBENCH_DURATION,
        "protocol": PGBENCH_PROTOCOL,
        "query_count": len(query_configs)
    },
    "performance_metrics": {
        "tps": tps,
        "total_transactions": len(latencies),
        "latency_p50_ms": p50 if latencies else None,
        "latency_p95_ms": p95 if latencies else None,
        "latency_p99_ms": p99 if latencies else None,
        "per_query": per_query,
        "cache_hit_pct": cache_hit_pct
    },
    "test_status": "completed" if res.returncode == 0 else "failed",
    "raw_output": res.stdout
}

# Save results to a file that can be accessed by the job
results_path = os.path.join(workdir, "pgbench_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n=== Test completed successfully! ===")
print(f"Results saved to: {results_path}")
print(json.dumps(results, indent=2))


# Return pgbench results to the job caller
# The results include both parsed metrics and the raw pgbench stdout
# which contains the summary stats we'll parse in the app

In [ ]:
dbutils.notebook.exit(json.dumps(results))